In [2]:
import gc
import random
from dataclasses import dataclass
from typing import List, Dict, Tuple, Optional
from collections import deque

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset, ConcatDataset
from torchvision import datasets, transforms


# ============================================================
# 0) Reproducibility / utilities
# ============================================================

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def cleanup_memory() -> None:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def accuracy(model: nn.Module, loader: DataLoader, device: torch.device) -> float:
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            logits = model(x)
            pred = logits.argmax(dim=1)
            correct += (pred == y).sum().item()
            total += y.size(0)
    return (correct / total) if total > 0 else 0.0


def fmt_table(df: pd.DataFrame) -> str:
    df2 = df.copy()
    for c in df2.columns:
        if pd.api.types.is_numeric_dtype(df2[c]):
            df2[c] = df2[c].map(lambda v: "–" if pd.isna(v) else f"{v:.3f}")
    return df2.to_string(index=False)


# ============================================================
# 1) Paper-aligned configuration (Table 4)
# ============================================================

@dataclass
class Table4Config:
    # FL setting (paper-style)
    K: int = 30                 # number of clients
    theta: float = 0.30         # gaming fraction
    R: int = 40                 # communication rounds
    E: int = 2                  # local epochs
    B: int = 64                 # batch size
    eta: float = 0.01           # learning rate
    mu: float = 0.9             # momentum

    # Non-IID partition (Dirichlet)
    alpha_dir: float = 0.5      # Dirichlet concentration

    # Public metric / leakage
    phi_leak: float = 1.0       # leak fraction of head-only public validation
    head: Tuple[int, ...] = (0, 1, 2, 3, 4)   # head classes (public metric)
    tail: Tuple[int, ...] = (5, 6, 7, 8, 9)   # tail classes (welfare)

    # Evaluation summary
    tail_window: int = 10       # average last L rounds (paper summary)

    # Reproducibility (Table 4 seeds; set to [42] if paper uses single seed)
    seeds: Tuple[int, ...] = (42,)

    # Data / device
    data_root: str = "./data"
    device: torch.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# ============================================================
# 2) Model (simple CNN)
# ============================================================

class SimpleCNN(nn.Module):
    def __init__(self, num_classes: int = 10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(inplace=True),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


def make_model(device: torch.device) -> nn.Module:
    return SimpleCNN(num_classes=10).to(device)


# ============================================================
# 3) Data preparation (Fashion-MNIST) + partitions + leak
# ============================================================

def make_dirichlet_partitions(
    labels: np.ndarray,
    K: int,
    alpha_dir: float,
    rng: np.random.Generator
) -> List[List[int]]:
    """
    Returns indices per client (indices refer to the LOCAL TRAIN subset index space).
    """
    n_classes = int(labels.max()) + 1
    client_indices = [[] for _ in range(K)]

    class_indices = []
    for k in range(n_classes):
        idx_k = np.where(labels == k)[0]
        rng.shuffle(idx_k)
        class_indices.append(idx_k)

    for k in range(n_classes):
        idx_k = class_indices[k]
        n_k = len(idx_k)
        if n_k == 0:
            continue

        proportions = rng.dirichlet(alpha_dir * np.ones(K))
        proportions = proportions / proportions.sum()
        splits = (np.cumsum(proportions) * n_k).astype(int)
        shards = np.split(idx_k, splits[:-1])

        for cid, shard in enumerate(shards):
            client_indices[cid].extend(shard.tolist())

    for cid in range(K):
        rng.shuffle(client_indices[cid])

    return client_indices


def prepare_fashionmnist(cfg: Table4Config, seed: int):
    """
    Creates:
      - client_datasets_base: list of Subset(train_local_set, indices_in_local_train_space)
      - public_val_loader (head-only)
      - hidden_tail_test_loader (tail-only)
      - leak_subset (head-only public val leak, as Subset(full_train, global_indices))
      - train_targets_local: labels for local-train index space
    """
    set_seed(seed)
    rng = np.random.default_rng(seed)

    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,))
    ])

    full_train = datasets.FashionMNIST(cfg.data_root, train=True, download=True, transform=transform)
    test_set = datasets.FashionMNIST(cfg.data_root, train=False, download=True, transform=transform)

    # 60k train: first 50k local, last 10k public-val candidates
    n_local = 50_000
    all_idx = np.arange(len(full_train))
    local_global_idx = all_idx[:n_local]           # indices into full_train
    pubval_global_idx = all_idx[n_local:]          # indices into full_train

    train_local_set = Subset(full_train, local_global_idx.tolist())

    # Labels in local-train index space (0..49999)
    train_targets_local = np.array(full_train.targets)[local_global_idx]

    # Public validation: head-only from candidate split
    full_train_targets = np.array(full_train.targets)
    head_mask_global = np.isin(full_train_targets, np.array(cfg.head, dtype=int))
    pubval_head_global_idx = np.array([i for i in pubval_global_idx if head_mask_global[i]], dtype=int)
    rng.shuffle(pubval_head_global_idx)

    leak_size = int(cfg.phi_leak * len(pubval_head_global_idx))
    leak_global_idx = pubval_head_global_idx[:leak_size]

    public_val_set = Subset(full_train, pubval_head_global_idx.tolist())
    public_val_loader = DataLoader(public_val_set, batch_size=cfg.B, shuffle=False, num_workers=0, pin_memory=torch.cuda.is_available())

    # Hidden welfare: tail-only from test set
    test_targets = np.array(test_set.targets)
    tail_mask = np.isin(test_targets, np.array(cfg.tail, dtype=int))
    tail_test_idx = np.where(tail_mask)[0]
    hidden_tail_test_set = Subset(test_set, tail_test_idx.tolist())
    hidden_tail_test_loader = DataLoader(hidden_tail_test_set, batch_size=cfg.B, shuffle=False, num_workers=0, pin_memory=torch.cuda.is_available())

    # Leak subset for gaming clients (head-only public)
    leak_subset = Subset(full_train, leak_global_idx.tolist())

    # Dirichlet partitions on local-train index space
    client_local_indices = make_dirichlet_partitions(
        labels=train_targets_local,
        K=cfg.K,
        alpha_dir=cfg.alpha_dir,
        rng=rng
    )
    client_datasets_base = [Subset(train_local_set, idxs) for idxs in client_local_indices]

    cleanup_memory()
    return client_datasets_base, public_val_loader, hidden_tail_test_loader, leak_subset, train_targets_local


def head_only_subset(
    base_subset: Subset,
    train_targets_local: np.ndarray,
    head_classes: Tuple[int, ...]
) -> Subset:
    """
    Fast head-only filtering in local-train index space.
    base_subset.indices are indices into LOCAL TRAIN subset (0..n_local-1).
    """
    idxs = np.array(base_subset.indices, dtype=int)
    keep = np.isin(train_targets_local[idxs], np.array(head_classes, dtype=int))
    kept = idxs[keep].tolist()
    return Subset(base_subset.dataset, kept)


# ============================================================
# 4) Local training + streaming FedAvg (memory-friendly)
# ============================================================

def local_train(
    model: nn.Module,
    dataset,
    E: int,
    B: int,
    eta: float,
    mu: float,
    device: torch.device
) -> None:
    loader = DataLoader(dataset, batch_size=B, shuffle=True, num_workers=0, pin_memory=torch.cuda.is_available())
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=eta, momentum=mu)

    model.train()
    for _ in range(E):
        for x, y in loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()


def fedavg_round_streaming(
    global_state_cpu: Dict[str, torch.Tensor],
    client_datasets,
    cfg: Table4Config,
) -> Dict[str, torch.Tensor]:
    """
    Streaming FedAvg:
      - For each client: train a local model starting from global_state_cpu
      - Accumulate weights on CPU (no list of models kept)
      - Return new global_state_cpu
    """
    device = cfg.device

    # Initialize sum_state on CPU
    sum_state = {k: torch.zeros_like(v, device="cpu") for k, v in global_state_cpu.items()}

    for ds in client_datasets:
        model = make_model(device)
        model.load_state_dict(global_state_cpu, strict=True)

        local_train(model, ds, cfg.E, cfg.B, cfg.eta, cfg.mu, device)

        sd = model.state_dict()
        with torch.no_grad():
            for k in sum_state.keys():
                sum_state[k] += sd[k].detach().to("cpu")

        # cleanup per client
        del model, sd
        cleanup_memory()

    # Average
    with torch.no_grad():
        for k in sum_state.keys():
            sum_state[k] /= float(cfg.K)

    cleanup_memory()
    return sum_state


# ============================================================
# 5) One run (aligned vs gaming) -> last-L means for W and M
# ============================================================

def run_one_setting(
    cfg: Table4Config,
    client_datasets_base,
    public_val_loader,
    hidden_tail_test_loader,
    leak_subset,
    train_targets_local: np.ndarray,
    gaming_frac: float
) -> Tuple[float, float]:
    """
    Returns:
      W_bar: mean of last-L rounds of tail-only hidden test accuracy (welfare W)
      M_bar: mean of last-L rounds of head-only public validation accuracy (metric M)
    """
    # Fixed client composition per run (deterministic given seed already set outside)
    K = cfg.K
    n_gaming = int(round(gaming_frac * K))
    rng_local = np.random.default_rng(0)  # deterministic selection given fixed seed context
    gaming_clients = set(rng_local.choice(np.arange(K), size=n_gaming, replace=False).tolist())

    # Build per-client training datasets
    client_train_datasets = []
    for cid in range(K):
        base_ds = client_datasets_base[cid]
        if (gaming_frac > 0.0) and (cid in gaming_clients):
            base_head = head_only_subset(base_ds, train_targets_local, cfg.head)
            ds = ConcatDataset([base_head, leak_subset])
        else:
            ds = base_ds
        client_train_datasets.append(ds)

    # Global model state on CPU
    global_model = make_model(cfg.device)
    global_state_cpu = {k: v.detach().to("cpu") for k, v in global_model.state_dict().items()}
    del global_model
    cleanup_memory()

    W_last = deque(maxlen=cfg.tail_window)
    M_last = deque(maxlen=cfg.tail_window)

    # R rounds
    for _ in range(cfg.R):
        global_state_cpu = fedavg_round_streaming(global_state_cpu, client_train_datasets, cfg)

        # Evaluate using a model instance
        model_eval = make_model(cfg.device)
        model_eval.load_state_dict(global_state_cpu, strict=True)

        W_t = accuracy(model_eval, hidden_tail_test_loader, cfg.device)
        M_t = accuracy(model_eval, public_val_loader, cfg.device)

        W_last.append(W_t)
        M_last.append(M_t)

        del model_eval
        cleanup_memory()

    W_bar = float(np.mean(W_last)) if len(W_last) > 0 else 0.0
    M_bar = float(np.mean(M_last)) if len(M_last) > 0 else 0.0

    # cleanup
    del client_train_datasets, global_state_cpu
    cleanup_memory()

    return W_bar, M_bar


# ============================================================
# 6) Table 4: aggregate over seeds and print ONLY the paper table
# ============================================================

def run_table_4(cfg: Table4Config) -> pd.DataFrame:
    """
    Outputs a 2-row table (Aligned vs Gaming) with columns:
      W, M, (M-W), PoG
    where W and M are means over last-L rounds, then averaged over seeds.
    """
    Ws_align, Ms_align = [], []
    Ws_game, Ms_game = [], []

    for seed in cfg.seeds:
        client_datasets_base, public_val_loader, hidden_tail_test_loader, leak_subset, train_targets_local = prepare_fashionmnist(cfg, seed)

        # aligned (theta=0)
        W_a, M_a = run_one_setting(
            cfg, client_datasets_base, public_val_loader, hidden_tail_test_loader,
            leak_subset, train_targets_local, gaming_frac=0.0
        )
        # gaming (theta)
        W_g, M_g = run_one_setting(
            cfg, client_datasets_base, public_val_loader, hidden_tail_test_loader,
            leak_subset, train_targets_local, gaming_frac=cfg.theta
        )

        Ws_align.append(W_a); Ms_align.append(M_a)
        Ws_game.append(W_g);  Ms_game.append(M_g)

        # cleanup seed-level objects
        del client_datasets_base, public_val_loader, hidden_tail_test_loader, leak_subset, train_targets_local
        cleanup_memory()

    W_align = float(np.mean(Ws_align))
    M_align = float(np.mean(Ms_align))
    W_game  = float(np.mean(Ws_game))
    M_game  = float(np.mean(Ms_game))

    PoG = np.nan if W_align <= 0 else (W_align - W_game) / W_align

    df = pd.DataFrame([
        {"Scenario": "Aligned", "W": W_align, "M": M_align, "M − W": (M_align - W_align), "PoG": np.nan},
        {"Scenario": "Gaming",  "W": W_game,  "M": M_game,  "M − W": (M_game  - W_game),  "PoG": PoG},
    ])
    return df


if __name__ == "__main__":
    cfg = Table4Config()
    # NOTE: No intermediate prints; Table 4 only.
    df4 = run_table_4(cfg)
    print("Table 4: Fashion-MNIST (head-metric M vs tail-welfare W), last-L-round mean; aligned vs gaming.")
    print(fmt_table(df4))

100%|██████████| 26.4M/26.4M [00:01<00:00, 17.7MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 303kB/s]
100%|██████████| 4.42M/4.42M [00:00<00:00, 5.62MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 11.2MB/s]


Table 4: Fashion-MNIST (head-metric M vs tail-welfare W), last-L-round mean; aligned vs gaming.
Scenario     W     M  M − W   PoG
 Aligned 0.885 0.879 -0.006     –
  Gaming 0.835 0.975  0.141 0.057
